# SPML Task 1 — Bonus Task
## MLP Autoencoder on Fashion-MNIST

### What is an Autoencoder?
An autoencoder is basically a neural network that compresses the input down to a smaller representation and then tries to recreate the original from that compressed version. It has two parts:
1. **Encoder** – squishes the input into a smaller latent vector (the bottleneck)
2. **Decoder** – tries to reconstruct the original image from that compressed version

The idea is that by forcing the network through a narrow bottleneck, it has to learn what's actually important in the data.

```
Input (784) --> Encoder --> Latent (z) --> Decoder --> Reconstructed (784)
```

### What this notebook does
- Builds a simple MLP-based autoencoder in PyTorch
- Trains it with 3 different latent sizes (8, 32, 128) to see how compression affects quality
- Plots loss curves and visualises reconstructions
- Does a quick t-SNE plot of the latent space (cool to see the clusters)
- Bonus: tests it as a denoiser
- Saves model weights using pickle


## 0. Imports & Setup


In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms

# set seeds so results are reproducible
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')


## 1. Load Fashion-MNIST

We don't actually need the class labels for training the autoencoder (it's unsupervised), but I'm keeping them around for the visualisation at the end.


In [ ]:
# hyperparameters
BATCH_SIZE = 128
EPOCHS = 30
LR = 1e-3
VAL_SPLIT = 0.1  # use 10% of training data for validation

CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# just normalise to [0,1] — keeping it in this range because the decoder
# uses sigmoid which also outputs [0,1], so the targets match
transform = transforms.Compose([
    transforms.ToTensor(),
])

# download the datasets
full_train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

# split off some data for validation
val_size = int(len(full_train_dataset) * VAL_SPLIT)
train_size = len(full_train_dataset) - val_size

train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f'Train: {train_size:,} samples')
print(f'Val:   {val_size:,} samples')
print(f'Test:  {len(test_dataset):,} samples')


## 2. Autoencoder Model Definition

The architecture is pretty straightforward — encoder gradually compresses the image from 784 down to the latent size, and the decoder mirrors it going back up to 784.

```
ENCODER: 784 -> 256 -> 128 -> 64 -> z
DECODER:   z -> 64  -> 128 -> 256 -> 784
```

I used ReLU activations in the hidden layers and sigmoid at the end of the decoder (since pixel values are in [0,1]).


In [ ]:
class Encoder(nn.Module):
    # compresses the image down to a latent vector of size latent_dim
    def __init__(self, latent_dim: int):
        super(Encoder, self).__init__()
        self.net = nn.Sequential(
            nn.Flatten(),          # flatten 28x28 image to 784
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, latent_dim)  # bottleneck, no activation here
        )

    def forward(self, x):
        return self.net(x)


class Decoder(nn.Module):
    # takes the latent vector and tries to reconstruct the image
    def __init__(self, latent_dim: int):
        super(Decoder, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64),  nn.ReLU(),
            nn.Linear(64, 128),         nn.ReLU(),
            nn.Linear(128, 256),        nn.ReLU(),
            nn.Linear(256, 784),        nn.Sigmoid()  # output in [0,1]
        )

    def forward(self, z):
        return self.net(z)


class MLP_Autoencoder(nn.Module):
    def __init__(self, latent_dim: int):
        super(MLP_Autoencoder, self).__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
        self.latent_dim = latent_dim

    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon, z

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)


# quick sanity check to make sure the shapes are right
dummy_model = MLP_Autoencoder(latent_dim=32).to(device)
dummy_input = torch.randn(4, 1, 28, 28).to(device)
recon_out, latent_out = dummy_model(dummy_input)
print(f'Input shape:  {dummy_input.shape}')
print(f'Latent shape: {latent_out.shape}')
print(f'Output shape: {recon_out.shape}')
del dummy_model, dummy_input, recon_out, latent_out


## 3. Training & Evaluation Helpers

Using MSE as the loss — we just want to minimise the pixel-level difference between the original and reconstructed image.


In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """runs one training epoch and returns the average loss"""
    model.train()
    total_loss = 0.0

    for images, _ in loader:  # we ignore labels
        images = images.to(device)

        optimizer.zero_grad()
        recon, _ = model(images)

        # flatten original to match decoder output
        target = images.view(images.size(0), -1)
        loss = criterion(recon, target)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_epoch(model, loader, criterion, device):
    """evaluates the model on a given loader, returns avg loss"""
    model.eval()
    total_loss = 0.0

    for images, _ in loader:
        images = images.to(device)
        recon, _ = model(images)
        target = images.view(images.size(0), -1)
        loss = criterion(recon, target)
        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


def train_autoencoder(latent_dim, epochs=EPOCHS, lr=LR, verbose=True):
    """builds and trains an autoencoder, returns the model and loss history"""
    print(f'\n--- Training with latent_dim = {latent_dim} ---')

    model = MLP_Autoencoder(latent_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # reduce lr if val loss stops improving
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=False
    )

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_ckpt_path = f'ae_best_z{latent_dim}.pt'

    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss = evaluate_epoch(model, val_loader, criterion, device)

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        # save checkpoint if this is the best val loss so far
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_ckpt_path)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'  Epoch {epoch}/{epochs}  |  Train: {train_loss:.6f}  |  Val: {val_loss:.6f}')

    # load the best weights before returning
    model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
    print(f'  Best val loss: {best_val_loss:.6f}')
    return model, history


## 4. Train with 3 Different Latent Dimensions

Trying 3 values to see how the bottleneck size affects reconstruction quality:
- **z=8**: very compressed, probably blurry
- **z=32**: hopefully a nice balance
- **z=128**: high quality but less compression


In [ ]:
LATENT_DIMS = [8, 32, 128]

results = {}  # store (model, history) for each latent dim

for z in LATENT_DIMS:
    model, history = train_autoencoder(latent_dim=z, epochs=EPOCHS, lr=LR)
    results[z] = (model, history)


## 5. Plot Training & Validation Loss Curves


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for ax, z in zip(axes, LATENT_DIMS):
    _, history = results[z]
    epochs_range = range(1, len(history['train_loss']) + 1)

    ax.plot(epochs_range, history['train_loss'], label='Train', marker='o',
            markersize=3, linewidth=1.5)
    ax.plot(epochs_range, history['val_loss'], label='Val', marker='s',
            markersize=3, linewidth=1.5, linestyle='--')

    # mark where the best val loss was
    best_ep = int(np.argmin(history['val_loss'])) + 1
    best_val = min(history['val_loss'])
    ax.axvline(best_ep, linestyle=':', color='gray', alpha=0.7)
    ax.annotate(f'Best: {best_val:.5f}\n@ ep {best_ep}',
                xy=(best_ep, best_val), xytext=(best_ep + 1, best_val * 1.05),
                fontsize=7, color='gray')

    ax.set_title(f'Latent dim = {z}', fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Train & Val Loss (MSE) vs Epochs', fontsize=13)
plt.tight_layout()
plt.savefig('ae_loss_curves.png', dpi=150)
plt.show()
print('saved to ae_loss_curves.png')


## 6. Original vs Reconstructed Images

Showing 10 test images per latent dim alongside their reconstructions. This is the most intuitive way to see the effect of the bottleneck size.


In [ ]:
@torch.no_grad()
def get_reconstructions(model, loader, n=10):
    """grab n images from the loader and return originals + reconstructions"""
    model.eval()
    images, labels = next(iter(loader))
    images = images[:n].to(device)

    recon, _ = model(images)
    recon = recon.view(-1, 28, 28).cpu().numpy()
    originals = images.squeeze(1).cpu().numpy()
    return originals, recon, labels[:n].numpy()


N_SHOW = 10

fig, axes = plt.subplots(
    nrows=len(LATENT_DIMS) * 2,
    ncols=N_SHOW,
    figsize=(N_SHOW * 1.4, len(LATENT_DIMS) * 2.8)
)

for i, z in enumerate(LATENT_DIMS):
    model, _ = results[z]
    originals, recons, lbls = get_reconstructions(model, test_loader, n=N_SHOW)

    row_orig = i * 2
    row_recon = i * 2 + 1

    for j in range(N_SHOW):
        ax_o = axes[row_orig, j]
        ax_o.imshow(originals[j], cmap='gray', vmin=0, vmax=1)
        ax_o.axis('off')
        if j == 0:
            ax_o.set_ylabel('Original', fontsize=8, labelpad=4)
        if i == 0:
            ax_o.set_title(CLASS_NAMES[lbls[j]], fontsize=6, pad=2)

        ax_r = axes[row_recon, j]
        ax_r.imshow(recons[j], cmap='gray', vmin=0, vmax=1)
        ax_r.axis('off')
        if j == 0:
            ax_r.set_ylabel(f'Recon z={z}', fontsize=8, labelpad=4)

plt.suptitle('Original vs Reconstructed (z=8, z=32, z=128)', fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('ae_reconstructions.png', dpi=150)
plt.show()
print('saved to ae_reconstructions.png')


## 7. Bottleneck Analysis

Comparing test MSE across all latent dims to see the quality vs compression tradeoff.


In [ ]:
criterion = nn.MSELoss()

print('Test MSE by latent dim:')
print('-' * 45)

test_losses = {}
for z in LATENT_DIMS:
    model, _ = results[z]
    test_loss = evaluate_epoch(model, test_loader, criterion, device)
    test_losses[z] = test_loss
    compression = 784 / z
    print(f'  z={z:<4}  MSE: {test_loss:.6f}  compression: {compression:.1f}x')

# bar chart for easy comparison
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    [str(z) for z in LATENT_DIMS],
    list(test_losses.values()),
    color=['#e74c3c', '#3498db', '#2ecc71'],
    width=0.5, edgecolor='black', linewidth=0.7
)

for bar, val in zip(bars, test_losses.values()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.0001,
        f'{val:.5f}',
        ha='center', va='bottom', fontsize=9
    )

ax.set_xlabel('Latent Dimension (z)', fontsize=11)
ax.set_ylabel('Test MSE Loss', fontsize=11)
ax.set_title('Reconstruction Quality vs Bottleneck Size', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('ae_bottleneck_comparison.png', dpi=150)
plt.show()
print('saved to ae_bottleneck_comparison.png')


## 8. Latent Space Visualisation (t-SNE on z=32)

Projecting the 32-d latent codes down to 2D with t-SNE to see if the encoder has learned meaningful structure. If it has, we should see class clusters even though labels were never used during training.


In [ ]:
from sklearn.manifold import TSNE

model_z32 = results[32][0]
model_z32.eval()

all_latents = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        z = model_z32.encode(images)
        all_latents.append(z.cpu().numpy())
        all_labels.append(labels.numpy())

all_latents = np.concatenate(all_latents, axis=0)  # (10000, 32)
all_labels = np.concatenate(all_labels, axis=0)

print(f'Latent codes shape: {all_latents.shape}')
print('Running t-SNE, might take a minute or two...')

tsne = TSNE(n_components=2, random_state=SEED, perplexity=40, n_iter=1000)
latents_2d = tsne.fit_transform(all_latents)

print('Done!')

fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.get_cmap('tab10', 10)

for class_idx in range(10):
    mask = all_labels == class_idx
    ax.scatter(
        latents_2d[mask, 0], latents_2d[mask, 1],
        s=4, alpha=0.5, color=cmap(class_idx),
        label=CLASS_NAMES[class_idx]
    )

ax.legend(markerscale=3, fontsize=9, loc='upper right')
ax.set_title('t-SNE of latent space (z=32) — coloured by class (not used in training!)', fontsize=11)
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('ae_tsne_latent_space.png', dpi=150)
plt.show()
print('saved to ae_tsne_latent_space.png')


## 9. Bonus: Denoising with the Autoencoder

A nice property of autoencoders is that they can act as denoisers — since the bottleneck forces the model to learn only the core structure, random noise can't really pass through. Let's test this by adding Gaussian noise and seeing how well z=32 cleans it up.


In [ ]:
NOISE_FACTOR = 0.4  # std of gaussian noise — increase to make it harder

model_z32.eval()
test_images, test_labels = next(iter(test_loader))
test_images = test_images[:8].to(device)

# add noise and clamp back to [0,1]
noisy_images = test_images + NOISE_FACTOR * torch.randn_like(test_images)
noisy_images = torch.clamp(noisy_images, 0.0, 1.0)

with torch.no_grad():
    denoised, _ = model_z32(noisy_images)

orig_np = test_images.squeeze(1).cpu().numpy()
noisy_np = noisy_images.squeeze(1).cpu().numpy()
denoised_np = denoised.view(-1, 28, 28).cpu().numpy()

# plot original, noisy, and denoised side by side
n = 8
fig, axes = plt.subplots(3, n, figsize=(n * 1.5, 5))

row_titles = ['Original', f'Noisy (sigma={NOISE_FACTOR})', 'Denoised (z=32)']
row_data = [orig_np, noisy_np, denoised_np]

for row_idx, (row_imgs, title) in enumerate(zip(row_data, row_titles)):
    for col_idx in range(n):
        ax = axes[row_idx, col_idx]
        ax.imshow(row_imgs[col_idx], cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
    axes[row_idx, 0].set_ylabel(title, fontsize=9, labelpad=4)

plt.suptitle('Denoising with Autoencoder (z=32)', fontsize=12)
plt.tight_layout()
plt.savefig('ae_denoising.png', dpi=150)
plt.show()
print('saved to ae_denoising.png')


## 10. Save Model Weights


In [ ]:
for z in LATENT_DIMS:
    model, _ = results[z]
    save_path = f'ae_weights_z{z}.pkl'

    with open(save_path, 'wb') as f:
        pickle.dump(model.state_dict(), f)

    # verify we can load it back properly
    with open(save_path, 'rb') as f:
        loaded_weights = pickle.load(f)

    verify = MLP_Autoencoder(z).to(device)
    verify.load_state_dict(loaded_weights)
    verify.eval()

    print(f'Saved & verified: {save_path}  ({os.path.getsize(save_path):,} bytes)')


## 11. Summary & Observations

### Results

| Latent dim | Compression | Test MSE | Quality |
|:---:|:---:|:---:|---|
| **8** | ~98x | Highest | Blurry, only basic shapes get through |
| **32** | ~24.5x | Medium | Pretty good, fine details start appearing |
| **128** | ~6x | Lowest | Almost identical to original |

### Takeaways

1. Smaller bottleneck = more compression but worse reconstruction quality (makes sense — less information to work with)
2. Even z=8 manages to capture the rough category of each item, which is impressive given how compressed it is
3. The t-SNE plot shows that the latent space is actually organised by class, even though the model never saw labels — cool that this emerges naturally
4. The denoising works better than I expected — the bottleneck acts as a natural filter for noise


In [ ]:
# check all output files got saved
output_files = [
    'ae_weights_z8.pkl', 'ae_weights_z32.pkl', 'ae_weights_z128.pkl',
    'ae_loss_curves.png', 'ae_reconstructions.png',
    'ae_bottleneck_comparison.png', 'ae_tsne_latent_space.png',
    'ae_denoising.png'
]

print('Output files:')
for fname in output_files:
    exists = os.path.exists(fname)
    size = os.path.getsize(fname) if exists else 0
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}]  {fname:<40}  {size:>10,} bytes')
